# LineScout · Milestone 2 GPU ingestion pipeline

Turn a folder of raw source artwork into a **validated LineScout gallery** plus
**feature shards**, on a free Colab GPU — no account, no paid tier, nothing to
install locally.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sehaan-1/drawable/blob/main/ml/colab/linescout_gpu_pipeline.ipynb)

**What you get at the end**

```
<DRIVE_ROOT>/gallery/<DATASET_VERSION>/
    originals/<asset_id>.png        source image, re-encoded as PNG
    line_art/<asset_id>.png         extracted (or native) line art
    thumbnails/<asset_id>.png       reference-panel tile
    manifest.json                   validated against ml/linescout_ml/manifest.py
    _pipeline/candidates.jsonl      resumable per-image state
    _pipeline/run_report.json       config, GPU, model versions, counts
<DRIVE_ROOT>/indexes/<DATASET_VERSION>/
    mobileclip2_s2/{index.json, shard-*.npz}
    dinov2_vits14/{index.json, shard-*.npz}
```

**How it is built.** This notebook is a thin driver: every stage lives in the
repository's `linescout_ml.colab` package, which is typed, linted, and unit
tested in CI. The cells below configure that package, call it, and show you what
came out. When a stage misbehaves, the fix belongs in a reviewed module, not in a
notebook cell.

| # | Stage | GPU | What it writes |
|---|---|---|---|
| 1 | Configure | — | paths, sources, toggles |
| 2 | Environment | — | Drive mount, repository, dependencies |
| 3 | Sanity check | no | full run on the committed fixture |
| 5 | Discover | no | candidates, asset ids, splits |
| 6 | **Extract line art** | **yes** | originals, line art, thumbnails |
| 7 | Measure | no | ink, text, quality, pHash, crop |
| 8 | De-duplicate | no | near-duplicate groups dropped |
| 9 | **Label** | **yes** | provisional style/scope, SFW gate |
| 10 | **Embed** | **yes** | MobileCLIP2 + DINOv2 shards |
| 11 | Build manifest | no | `manifest.json`, validated |
| 12 | Export | no | zip, Drive copy, run report |

> **Budget.** On a T4, extraction runs at roughly 1-2 images/second at 1024 px
> and embedding at ~50 images/second, so allow about 25 minutes per 1,000 images
> for the whole pipeline. Every stage persists before the next one starts, so a
> disconnect costs you the in-flight batch, not the run.

> **Licences.** The notebook downloads public model weights and writes licence
> identifiers into a provenance manifest. It refuses to run with a placeholder
> licence, and section 15 spells out what each model's terms actually are —
> including the fact that **MobileCLIP2 weights are research-only**. Read that
> before redistributing anything this pipeline produces.

## 0 · Before you start

1. **Runtime → Change runtime type → T4 GPU.** Cell 2 prints what you actually
   got. A CPU runtime still works end to end; extraction is simply ~30x slower.
2. **Upload your sources to Drive**, one folder per dataset:
   ```
   MyDrive/LineScout/sources/
       amateur_drawings/…
       manga109/<title>/page-001.png
       met_openaccess/…
   ```
   Folders are scanned recursively. `.png .jpg .jpeg .webp .bmp .gif` are picked
   up; everything else is ignored.
3. **Know your licence.** Each source needs a `license_id` you verified yourself.
   Several datasets this project targets (Manga109, eBDtheque, Human-Art) require
   an access application and forbid redistribution — start those early and record
   exactly what you were granted.
4. **No tokens, no secrets.** Weights come from public Hugging Face repos and
   `dl.fbaipublicfiles.com`; the repository clone is public.

## 1 · Configuration

The only cell you normally need to edit.

In [ ]:
# @title 1 · Run configuration (edit me — Colab renders this cell as a form)
from pathlib import Path

DATASET_VERSION = "2026.09.06-colab1"  # @param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/LineScout"  # @param {type:"string"}
MOUNT_DRIVE = True  # @param {type:"boolean"}
REPO_URL = "https://github.com/Sehaan-1/drawable.git"  # @param {type:"string"}
REPO_DIR = ""  # @param {type:"string"}  # blank = Drive copy if present, else /content/drawable

# One entry per raw dataset. `preset` keys are printed by cell 4; `license_id` is
# mandatory and must be something you verified — the pipeline refuses to write a
# provenance manifest with a placeholder in it.
SOURCES = [
    {
        "preset": "amateur_drawings",
        "folder": "amateur_drawings",  # relative to SOURCES_ROOT, or an absolute "root"
        "license_id": "REPLACE-ME",
        "extractor": "",  # optional: anime2sketch | informative_drawings
        "sfw_method": "",  # optional: source_rating | opennsfw2 | manual
        "work_grouping": "",  # optional: filename | parent_dir
    },
]

LIMIT_PER_SOURCE = 0  # @param {type:"integer"}  # 0 = no limit; try 20 for a first smoke run
LINE_ART_RESOLUTION = 1024  # @param {type:"slider", min:256, max:2048, step:64}
THUMBNAIL_SIZE = 256  # @param {type:"integer"}
BATCH_SIZE = 8  # @param {type:"slider", min:1, max:32, step:1}
DEVICE = "auto"  # @param ["auto", "cuda", "cpu"]
RUN_EXTRACTION = True  # @param {type:"boolean"}
RUN_DEDUPE = True  # @param {type:"boolean"}
RUN_LABELS = True  # @param {type:"boolean"}
RUN_EMBEDDINGS = True  # @param {type:"boolean"}
EMBEDDERS = "mobileclip2_s2,dinov2_vits14"  # @param {type:"string"}
LABELER_MODEL = "MobileCLIP2-S2"  # @param {type:"string"}
EXPORT_TO_DRIVE = True  # @param {type:"boolean"}
DOWNLOAD_ZIP = True  # @param {type:"boolean"}

SOURCES_ROOT = Path(DRIVE_ROOT) / "sources"
GALLERY_ROOT = Path(DRIVE_ROOT) / "gallery" / DATASET_VERSION
INDEX_ROOT = Path(DRIVE_ROOT) / "indexes" / DATASET_VERSION
ZIP_PATH = Path("/content") / f"linescout-{DATASET_VERSION}.zip"

print(f"dataset version : {DATASET_VERSION}")
print(f"sources         : {SOURCES_ROOT}")
print(f"gallery         : {GALLERY_ROOT}")
print(f"indexes         : {INDEX_ROOT}")
print(f"sources declared: {len(SOURCES)}")

## 2 · Environment

Mounts Drive, finds (or clones) the repository, and installs **only what is
missing**. Two deliberate choices:

* **torch is never reinstalled.** Colab ships a CUDA build matched to the host
  driver; letting pip resolve it again is the classic way to end up with a
  CPU-only torch or a broken cuDNN. The installed versions are written to a
  constraints file and passed to every install, so dependencies can be added
  without touching torch.
* **The repository is imported, not vendored.** The manifest schema, the
  taxonomy, and every stage come from `linescout_ml`, so notebook and repository
  cannot drift apart.

In [ ]:
# @title 2 · GPU, Drive, repository, dependencies
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

# --- what hardware did we actually get? --------------------------------------
probe = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
print(probe.stdout.strip() or "nvidia-smi not found — this is a CPU runtime")

# --- Drive -------------------------------------------------------------------
if MOUNT_DRIVE and importlib.util.find_spec("google.colab") is not None:
    from google.colab import drive

    drive.mount("/content/drive")
elif MOUNT_DRIVE:
    print("not running in Colab; skipping the Drive mount")


# --- repository --------------------------------------------------------------
def find_repo() -> Path:
    """Drive copy first (survives runtime restarts), otherwise clone."""
    explicit = Path(os.environ.get("LINESCOUT_REPO") or REPO_DIR or "/nonexistent")
    candidates = [explicit, Path(DRIVE_ROOT).parent / "drawable", Path("/content/drawable")]
    for candidate in candidates:
        if (candidate / "ml" / "linescout_ml").is_dir():
            print(f"using repository at {candidate}")
            return candidate
    target = Path("/content/drawable")
    print(f"cloning {REPO_URL} -> {target}")
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(target)])
    return target


REPO = find_repo()
sys.path.insert(0, str(REPO / "ml"))


# --- dependencies ------------------------------------------------------------
CONSTRAINTS = Path("/tmp/linescout-constraints.txt")


def pin_torch() -> list[str]:
    """Freeze whatever torch Colab gave us so pip cannot swap it out."""
    lines = []
    for name in ("torch", "torchvision"):
        if importlib.util.find_spec(name) is None:
            continue
        version = str(importlib.import_module(name).__version__).split("+")[0]
        lines.append(f"{name}=={version}")
    CONSTRAINTS.write_text("\n".join(lines) + "\n")
    return lines


def pip(*args: str) -> None:
    command = [sys.executable, "-m", "pip", "install", "--quiet"]
    if CONSTRAINTS.is_file():
        command += ["-c", str(CONSTRAINTS)]
    subprocess.check_call([*command, *args])


def have(module: str) -> bool:
    return importlib.util.find_spec(module) is not None


print("pinned:", pin_torch() or "no torch installed (CPU runtime)")

BASE = {"pydantic": "pydantic>=2.9,<3", "PIL": "pillow>=10.4", "numpy": "numpy>=1.26"}
missing = [spec for module, spec in BASE.items() if not have(module)]
if missing:
    pip(*missing)
if RUN_EXTRACTION and not have("controlnet_aux"):
    pip("controlnet-aux>=0.0.9")
if (RUN_LABELS or RUN_EMBEDDINGS) and not have("open_clip"):
    pip("open-clip-torch>=3.1", "timm>=1.0")
needs_nsfw = any("opennsfw2" in str(source.get("sfw_method", "")) for source in SOURCES)
if needs_nsfw and not have("opennsfw2"):
    pip("opennsfw2>=0.16")  # needs TensorFlow, which Colab ships
for module, spec in (("tqdm", "tqdm"), ("matplotlib", "matplotlib")):
    if not have(module):
        pip(spec)

import torch  # noqa: E402

import linescout_ml.colab as pipeline  # noqa: E402

cuda = torch.cuda.is_available()
device_name = torch.cuda.get_device_name(0) if cuda else "cpu only"
print(f"python        {sys.version.split()[0]}")
print(f"torch         {torch.__version__}   cuda={cuda}   {device_name}")
print(f"linescout_ml  {REPO / 'ml'}")
print(f"extractors    {pipeline.extraction_versions()}")

## 3 · Sanity check (no GPU, ~10 seconds)

Runs the **whole pipeline** against the synthetic fixture committed to the
repository — the same 24 generated drawings the API serves in fixture mode. No
detector, no CLIP, no embeddings. It proves the plumbing (discovery → normalised
line art → measurements → de-duplication → manifest → zip) works in *this*
runtime before you spend GPU minutes on real data.

The fixture deliberately contains repeated pages, so de-duplication keeps about
10 of the 24 candidates. That is the stage working, not data loss.

Skip this cell once you have seen it pass.

In [ ]:
# @title 3 · Dry run on the committed synthetic fixture
from linescout_ml.colab import PipelineConfig, PipelineRunner, preset_source

FIXTURE = REPO / "ml" / "fixtures" / "synthetic" / "originals"
DRY_ROOT = Path("/content/linescout-dryrun")

dry_config = PipelineConfig(
    dataset_version=DATASET_VERSION,
    output_root=DRY_ROOT / "gallery",
    embeddings_root=DRY_ROOT / "indexes",
    sources=[preset_source("synthetic", root=FIXTURE, license_id="synthetic-fixture")],
    label=False,
    embed=False,
    device="cpu",
    line_art_resolution=512,
    thumbnail_size=THUMBNAIL_SIZE,
)
dry_report = PipelineRunner(dry_config, progress=lambda stage, done, total: None).run_all(
    zip_path=DRY_ROOT / "dryrun.zip"
)

counts = dry_report["summary"]["candidates"]
print(
    f"candidates : {counts['total']} found, {counts['active']} kept,"
    f" {counts['duplicates']} duplicates, {counts['skipped']} skipped"
)
print(f"records    : {dry_report['summary']['total']}")
print(
    f"zip        : {dry_report['outputs']['zip']['path']}"
    f" ({dry_report['outputs']['zip']['bytes']} bytes)"
)
print(f"sha256     : {dry_report['outputs']['zip']['sha256'][:16]}…")
print("stages     :", ", ".join(stage["name"] for stage in dry_report["stages"]))

## 4 · Build the run configuration

Expands your `SOURCES` entries into typed `SourceSpec` objects. This cell is the
guardrail: a placeholder licence stops the run here rather than three hours
later, and a missing Drive folder is reported before anything is written.

Run it once to see the presets and what each one assumes about origin, extractor,
and licence.

In [ ]:
# @title 4 · Presets, sources, and the PipelineConfig
from pydantic import ValidationError

from linescout_ml.colab import PipelineConfig, SourceSpec, preset_source, preset_table

print(f"{'preset':<24} licence you must verify")
for row in preset_table():
    print(f"{row['key']:<24} {row['license_note']}")
    print(f"{'':<24} ↳ {row['description']}")

PLACEHOLDER_LICENSES = {"", "REPLACE-ME", "TBD", "UNKNOWN", "N/A"}


def build_sources() -> list[SourceSpec]:
    """Turn the SOURCES form data into validated SourceSpec objects."""
    specs: list[SourceSpec] = []
    for entry in SOURCES:
        preset = str(entry.get("preset", "")).strip()
        if not preset:
            raise SystemExit(f"every SOURCES entry needs a 'preset' key; got {entry}")
        license_id = str(entry.get("license_id", "")).strip()
        if license_id.upper() in PLACEHOLDER_LICENSES:
            raise SystemExit(
                f"source '{preset}' needs a real license_id. The manifest is a provenance\n"
                "document: record the licence you were actually granted (for example\n"
                "'CC0-1.0', 'cc-by-nc-4.0', 'manga109-research-only'), not a placeholder."
            )
        overrides = {
            key: str(entry[key])
            for key in ("extractor", "sfw_method", "work_grouping")
            if str(entry.get(key, "")).strip()
        }
        if entry.get("root"):
            root = Path(str(entry["root"]))
        else:
            root = SOURCES_ROOT / str(entry.get("folder") or preset)
        try:
            spec = preset_source(
                preset, root=root, license_id=license_id, name=entry.get("name"), **overrides
            )
        except ValidationError as error:
            # For example: an extractor override on a native-line-art source.
            raise SystemExit(f"source '{preset}' is not a valid combination:\n{error}") from None
        except ValueError as error:
            raise SystemExit(f"source '{preset}': {error}") from None
        specs.append(spec)
    if not specs:
        raise SystemExit("SOURCES is empty — add at least one dataset to process.")
    return specs


source_specs = build_sources()

CONFIG = PipelineConfig(
    dataset_version=DATASET_VERSION,
    output_root=GALLERY_ROOT,
    embeddings_root=INDEX_ROOT,
    sources=source_specs,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    limit_per_source=LIMIT_PER_SOURCE or None,
    thumbnail_size=THUMBNAIL_SIZE,
    line_art_resolution=LINE_ART_RESOLUTION,
    extract_line_art=RUN_EXTRACTION,
    dedupe=RUN_DEDUPE,
    label=RUN_LABELS,
    embed=RUN_EMBEDDINGS,
    embedders=[key.strip() for key in EMBEDDERS.split(",") if key.strip()],
    labeler_model=LABELER_MODEL,
)

print(f"\n{len(source_specs)} source(s) -> {CONFIG.output_root}")
for spec in source_specs:
    present = "ok" if spec.root.is_dir() else "MISSING"
    print(
        f"  [{present:>7}] {spec.name:<20} origin={spec.origin.value:<19}"
        f" extractor={spec.extractor:<20} sfw={spec.sfw_method:<24}"
        f" licence={spec.license_id}"
    )
    print(f"            {spec.root}")

In [ ]:
# @title 4b · Runner: progress bars and live notes
from tqdm.auto import tqdm

from linescout_ml.colab import PipelineRunner

_bars: dict[str, tqdm] = {}


def _progress(stage: str, done: int, total: int) -> None:
    bar = _bars.get(stage)
    if bar is None or bar.total != total:
        if bar is not None:
            bar.close()
        bar = tqdm(total=total, desc=stage, unit="img", leave=True)
        _bars[stage] = bar
    bar.n = min(done, total or 0)
    bar.refresh()


def _note(message: str) -> None:
    print(f"  note: {message}")


def close_bars() -> None:
    for bar in _bars.values():
        bar.close()
    _bars.clear()


RUNNER = PipelineRunner(CONFIG, progress=_progress, on_note=_note)
print(f"device : {RUNNER.device}")
print(f"gallery: {CONFIG.output_root}")
print(f"state  : {CONFIG.candidates_path}")

## 5 · Discover

Walks each source folder, assigns the deterministic asset id
(`ls_<dataset>_<sha256 prefix>`), groups images into *source works*, and derives
each work's split from a hash. That is what makes the manifest rule "assets from
one source work never cross splits" true by construction rather than by post-hoc
checking.

State is written to `_pipeline/candidates.jsonl` after every stage, so a
restarted runtime resumes instead of starting over.

In [ ]:
# @title 5 · Discover sources
from collections import Counter

from linescout_ml.colab import candidate_summary

discover_stage = RUNNER.discover()
close_bars()

print(f"candidates : {discover_stage.processed}")
print(f"state file : {RUNNER.config.candidates_path}")
for note in discover_stage.notes:
    print(f"  note: {note}")
print(f"summary    : {candidate_summary(RUNNER.store)}")
print(f"splits     : {dict(Counter(c.split.value for c in RUNNER.store))}")

## 6 · Extract line art · *GPU*

Runs the extractor each source asked for — `anime2sketch` (Anime2Sketch weights)
for manga and anime, `informative_drawings` (Chan *et al.*) for photographs,
paintings, and academic drawing. Both are served by `controlnet_aux` from
Hugging Face, so there is no Google Drive checkpoint hunting. Sources marked
`native_line_art` are normalised (alpha flattened onto white, EXIF orientation
applied) and never re-drawn.

Three files per asset are written here: the PNG original, the line art, and the
thumbnail. `controlnet_aux` works on a multiple of 64, so the line art is resized
back to the source geometry — the API serves line art and thumbnail side by side,
and the crop overlay assumes they line up pixel for pixel.

If CUDA runs out of memory on one huge scan, the runner clears the cache and
retries that image at half resolution instead of ending the run.

In [ ]:
# @title 6 · Extract line art (GPU)
extract_stage = RUNNER.run_extract()
close_bars()

print(f"processed : {extract_stage.processed}")
print(f"skipped   : {extract_stage.skipped} (already on disk from an earlier run)")
print(f"failed    : {extract_stage.failed}")
print(f"seconds   : {extract_stage.seconds:.1f}")
for note in extract_stage.notes:
    print(f"  note: {note}")

In [ ]:
# @title 6b · Compare a few originals with their line art
import matplotlib.pyplot as plt
from PIL import Image

from linescout_ml.colab import asset_paths


def show_pairs(limit: int = 6) -> None:
    """Grid of original | line art for the first `limit` extracted candidates."""
    ready = [c for c in RUNNER.store.active if c.line_art_path][:limit]
    if not ready:
        print("nothing extracted yet")
        return
    root = RUNNER.config.output_root
    figure, axes = plt.subplots(len(ready), 2, figsize=(8, 4 * len(ready)), squeeze=False)
    for row, candidate in enumerate(ready):
        paths = asset_paths(candidate.asset_id)
        axes[row, 0].imshow(Image.open(root / paths.original))
        axes[row, 0].set_title(
            f"original · {candidate.width}×{candidate.height} · {candidate.item_id[:32]}",
            fontsize=8,
        )
        axes[row, 1].imshow(Image.open(root / candidate.line_art_path), cmap="gray")
        axes[row, 1].set_title(
            f"{candidate.extraction_model or 'native'}@{candidate.extraction_version or '-'}",
            fontsize=8,
        )
        for axis in axes[row]:
            axis.axis("off")
    figure.tight_layout()
    plt.show()


show_pairs()

## 7 · Measure · *CPU*

Computes every number the manifest needs but no human should have to type, on a
view downscaled to 512 px so cost is flat regardless of source resolution:

* `ink_coverage` — pixels darker than 200/255, the same threshold the API's query
  preprocessing uses, so a stored asset and a query sketch are measured the same
  way
* `text_coverage` — share of the frame taken by glyph-like runs (speech bubbles,
  captions, watermarks, signatures); a connected-component heuristic, **not** OCR
* `quality_score` — a fixed weighted blend of resolution, ink density, paper
  cleanliness, and text contamination. It orders the curation queue; it does not
  judge art, and curation overrides it
* `phash` — 64-bit DCT perceptual hash, bit-for-bit compatible with
  `imagehash.phash`
* `crop` — the ink bounding box with 10 % padding, scaled back to source pixels,
  as the reviewer's starting frame

In [ ]:
# @title 7 · Measure line art
import numpy as np

measure_stage = RUNNER.run_measure()
close_bars()

print(f"processed : {measure_stage.processed}")
print(f"skipped   : {measure_stage.skipped}")
print(f"failed    : {measure_stage.failed}")
print(f"seconds   : {measure_stage.seconds:.1f}")
for note in measure_stage.notes:
    print(f"  note: {note}")

measured = [c.measurements for c in RUNNER.store.active if c.measurements]
if measured:
    for field in ("ink_coverage", "text_coverage", "quality_score"):
        values = np.array([getattr(item, field) for item in measured], dtype=float)
        print(
            f"{field:<15} min={values.min():.4f} median={np.median(values):.4f}"
            f" max={values.max():.4f}"
        )
    texty = [
        c for c in RUNNER.store.active if c.measurements and c.measurements.text_coverage > 0.02
    ]
    print(f"text > 2%   : {len(texty)} candidate(s) worth a look before curation")
    print(f"skipped     : {len([c for c in RUNNER.store if c.skip_reason])} candidate(s)")

## 8 · De-duplicate · *CPU*

Groups candidates whose pHash Hamming distance is within `dedupe_threshold`
(default 6) and keeps the lowest candidate key in each group. Duplicates are
marked in the state file, so they never reach the manifest, never get embedded,
and never appear in the export zip — but their files stay on disk, so you can
change your mind cheaply.

Marks are recomputed from scratch every time this cell runs. Too aggressive?

```python
RUNNER.config.dedupe_threshold = 3   # stricter, keeps more assets
RUNNER.config.dedupe = False         # or turn the stage off entirely
```

…then re-run this cell and everything you restore flows into the manifest again.
Scraped sources are full of rescaled re-uploads, so this is the cheapest quality
win in the pipeline — and it runs before you pay for labelling and embeddings.

In [ ]:
# @title 8 · De-duplicate by perceptual hash
from linescout_ml.colab import candidate_summary

dedupe_stage = RUNNER.run_dedupe()
close_bars()

print(f"duplicates removed : {dedupe_stage.processed}")
print(f"state              : {candidate_summary(RUNNER.store)}")
for note in dedupe_stage.notes:
    print(f"  note: {note}")

duplicates = [c for c in RUNNER.store if c.duplicate_of]
for candidate in duplicates[:10]:
    print(f"  {candidate.key}  ==  {candidate.duplicate_of}")
if len(duplicates) > 10:
    print(f"  … and {len(duplicates) - 10} more")

## 9 · Label · *GPU* (optional)

Asks the MobileCLIP2 text encoder to rank the five style families and the eight
scope buckets for each asset, using several prompts per label. The winner is
written as **provisional**: `labels.labelled_by` records whether a CLIP encoder
ranked it or the source default was used, the raw probabilities are stored
alongside, and the curation UI exists to correct all of it.

The SFW gate is separate and deliberately cheap. Sources whose terms already
guarantee SFW content are recorded as `source_rating`; only scraped sources pay
for `opennsfw2` — which screens the **original**, not the line art, because
extraction removes exactly the content a classifier needs to see. An asset that
fails the gate is written `quarantined` + `enabled=false`, which the manifest
enforces as an invariant.

Set `RUN_LABELS = False` in cell 1 to use source defaults instead (no GPU work).

In [ ]:
# @title 9 · Zero-shot labels + SFW gate
from collections import Counter

label_stage = RUNNER.run_label()
close_bars()

print(f"processed : {label_stage.processed}")
print(f"skipped   : {label_stage.skipped}")
print(f"failed    : {label_stage.failed}")
for note in label_stage.notes:
    print(f"  note: {note}")

labelled = [c.labels for c in RUNNER.store.active if c.labels]
if labelled:
    print("style      :", dict(Counter(item.primary_style.value for item in labelled)))
    print("scope      :", dict(Counter(s.value for item in labelled for s in item.scopes)))
    print("labelled_by:", dict(Counter(item.labelled_by.value for item in labelled)))
    print(
        "sfw        :",
        dict(
            Counter(
                f"{item.sfw.method}:{'safe' if item.sfw.safe else 'UNSAFE'}" for item in labelled
            )
        ),
    )
    unsafe = [c for c in RUNNER.store.active if c.labels and not c.labels.sfw.safe]
    if unsafe:
        print(f"quarantined: {len(unsafe)} asset(s) will be written disabled")

## 10 · Embed · *GPU*

Writes L2-normalised feature vectors into resumable `.npz` shards:

* **MobileCLIP2-S2** (512-d) — text-aligned, and the same encoder that produced
  the labels, so the model is loaded once
* **DINOv2 ViT-S/14** (384-d) — self-supervised shape features. Line art has no
  colour and no texture, which is exactly where a text-aligned encoder is
  weakest, so the two are complementary; Milestone 4 concatenates them

`index.json` beside the shards records the model card, dimension, and licence of
whatever produced them, so an index can be rebuilt later without guessing which
checkpoint was in use. Re-running the cell embeds only the assets that are
missing — which is what makes a disconnect survivable.

In [ ]:
# @title 10 · Feature shards
import json

embed_stage = RUNNER.run_embed()
close_bars()

print(f"embedded : {embed_stage.processed}")
print(f"skipped  : {embed_stage.skipped} (already in the index)")
print(f"failed   : {embed_stage.failed}")
for note in embed_stage.notes:
    print(f"  note: {note}")

for key in CONFIG.embedders:
    index_path = CONFIG.resolved_embeddings_root() / key / "index.json"
    if not index_path.is_file():
        print(f"\n{key}: no index written")
        continue
    index = json.loads(index_path.read_text())
    print(
        f"\n{key}: {index['count']} vectors x {index['spec']['dim']}d"
        f" in {len(index['shards'])} shard(s)"
    )
    print(f"  model   : {index['spec']['name']} ({index['spec']['upstream']})")
    print(f"  licence : {index['spec']['license']}")

## 11 · Build the manifest · *CPU*

Assembles one `ManifestRecord` per surviving candidate and validates the whole
collection — including the "one source work, one split" rule — before writing
`manifest.json`. If a previous manifest exists it is **merged**, not replaced:
incoming records win on `asset_id` and existing records are preserved, so you can
add a second dataset next week without rebuilding the first.

The build refuses to write a manifest whose enabled assets are missing files,
because that is precisely what `services/api` would refuse to start against.

In [ ]:
# @title 11 · Manifest slice
from linescout_ml.colab import missing_files, summarise

MANIFEST = RUNNER.run_build()
close_bars()

summary = summarise(MANIFEST.records)
print(f"records        : {len(MANIFEST.records)}")
print(f"enabled        : {len(MANIFEST.enabled_records)}")
print(f"dataset_version: {MANIFEST.dataset_version}")
print(f"content_hash   : {MANIFEST.content_hash()[:16]}…")
print(f"missing files  : {len(missing_files(MANIFEST, CONFIG.output_root))}")
print(f"\nby style : {summary['by_style']}")
print(f"by scope : {summary['by_scope']}")
print(f"by split : {summary['by_split']}")
print(f"by origin: {summary['by_origin']}")
print(f"review   : {summary['by_review_state']}")
print(
    f"quality  : min={summary['quality_min']} median={summary['quality_median']}"
    f" max={summary['quality_max']}"
)

In [ ]:
# @title 11b · Validate it the way the project CLI does
from linescout_ml.cli import main as linescout_manifest_cli

exit_code = linescout_manifest_cli(["validate", str(CONFIG.manifest_path), "--require-files"])
print("exit code:", exit_code)
assert exit_code == 0, "the gallery manifest failed validation — do not export it"

## 12 · Export

Zips exactly the files the manifest vouches for (plus the manifest and the run
report), optionally mirrors the gallery to Drive, and starts a browser download.

Prefer the zip over the Drive copy for anything large: a gallery is tens of
thousands of small files, and Drive's FUSE mount is slow and prone to partial
writes. The zip's SHA-256 is recorded in the run report, so you can verify it
after the download.

In [ ]:
# @title 12 · Zip, Drive copy, run report
outputs = RUNNER.run_export(
    zip_path=ZIP_PATH if DOWNLOAD_ZIP or EXPORT_TO_DRIVE else None,
    drive_root=Path(DRIVE_ROOT) / "gallery-export" if EXPORT_TO_DRIVE else None,
    download=DOWNLOAD_ZIP,
)
close_bars()

if "zip" in outputs:
    print(f"zip    : {outputs['zip']['path']}")
    print(f"size   : {outputs['zip']['bytes']} bytes")
    print(f"sha256 : {outputs['zip']['sha256']}")
if "drive" in outputs:
    print(f"drive  : {outputs['drive']['path']}")
print(f"report : {outputs['run_report']}")

In [ ]:
# @title 12b · Run report — keep this file next to the dataset
import json

report_path = RUNNER.config.state_dir / "run_report.json"
report = json.loads(report_path.read_text())

print(json.dumps({key: report[key] for key in ("created_at", "gpu", "embedders")}, indent=2))
print("\nsummary:")
for key, value in report["summary"].items():
    print(f"  {key:<10} {value}")
print("\nstages:")
for stage in report["stages"]:
    print(
        f"  {stage['name']:<9} processed={stage['processed']:<6}"
        f" skipped={stage['skipped']:<6} failed={stage['failed']:<4}"
        f" {stage['seconds']:>7.1f}s"
    )

### Hands-off alternative

Prefer one cell? `run_all` chains every enabled stage in order and returns the
run report. Use it after you have watched the stages run once, so you know what
the numbers should look like.

```python
report = PipelineRunner(CONFIG, progress=_progress, on_note=_note).run_all(
    zip_path=ZIP_PATH,
    drive_root=Path(DRIVE_ROOT) / "gallery-export",
    download=True,
)
```

## 13 · Back on your machine

1. Unzip the gallery into the repository's ignored data directory:
   ```bash
   unzip linescout-<version>.zip -d data/gallery/<version>/
   # feature shards (Milestone 4 index building) go to:
   #   data/indexes/<version>/{mobileclip2_s2,dinov2_vits14}/
   ```
2. Point the API at it and turn on curation — in `services/api/.env`:
   ```bash
   LINESCOUT_GALLERY_MANIFEST=data/gallery/<version>/manifest.json
   LINESCOUT_CURATION_MODE=1
   ```
3. Validate, then boot:
   ```bash
   npm run setup:py
   ml/.venv/bin/linescout-manifest validate data/gallery/<version>/manifest.json --require-files
   npm run dev:all          # API on :8000, web on :5173
   ```
4. Curate at `http://127.0.0.1:5173/curate`. Every asset arrives
   `review.state="unreviewed"` with `enabled=true`, which is what lets the
   curation UI render it; rejecting an asset flips `enabled` off.
5. When the batch is done, `POST /api/v1/curation/snapshots` writes an immutable
   label snapshot — the artefact a later manifest re-export patches from.

**Nothing here needs to run twice.** Candidate state, manifest merge, and the
embedding index are all idempotent: re-run any cell and it picks up where it
stopped.

## 14 · Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `nvidia-smi not found` | CPU runtime | Runtime → Change runtime type → T4 GPU. Everything still runs; extraction is ~30x slower |
| `CUDA out of memory` | one huge scan | the runner already retries that image at half resolution; lower `LINE_ART_RESOLUTION` or `BATCH_SIZE` if it recurs |
| `RuntimeError: … disconnected` | Colab recycled the VM | re-run cells 2, 4, 4b, then the stage you were in — `candidates.jsonl` and the embedding index make it resume |
| `MissingDependencyError` | a stage's package is absent | cell 2 installs only what the toggles need; enable the toggle and re-run cell 2 |
| `source 'x' needs a real license_id` | placeholder licence | intentional — put the licence you were actually granted in `SOURCES` |
| `FileNotFoundError: … root does not exist` | Drive path typo | check `SOURCES_ROOT`; Drive mounts at `/content/drive/MyDrive/…` |
| `git clone` fails | private fork | set `REPO_DIR` to a Drive copy, or upload a zip and unzip it to `/content/drawable` |
| `gallery is incomplete` | a file vanished mid-run | re-run cell 6; missing outputs are rewritten, existing ones skipped |
| `queue_empty` in the curation UI | everything already reviewed | run `scripts/reset_reviews.py` against a copy of the database |
| Line art looks like grey mush | wrong extractor for the source | `anime2sketch` for manga/anime, `informative_drawings` for photos and paintings; set `extractor` per source in `SOURCES` |

**Free-tier reality check.** Colab disconnects idle and heavy sessions, so the
pipeline is built for that: every stage persists before the next one starts,
embeddings are sharded, and the manifest is merged rather than rewritten. Treat a
run as a series of resumable sessions, not one long transaction.

## 15 · Attribution and licences

The manifest records a licence for every **asset**. This table is about the
**models** whose weights ran to produce them. Both matter, and only one of them
is comfortable.

| Component | Used for | Licence | Note |
|---|---|---|---|
| [Anime2Sketch](https://github.com/Mukosame/Anime2Sketch) (`netG.pth`) | line art from manga/anime | MIT | weights mirrored on HF as `lllyasviel/Annotators` |
| [Informative Drawings](https://github.com/carolineec/informative-drawings) (`sk_model.pth`) | line art from photos/paintings | MIT | Chan *et al.*, CVPR 2022 |
| [controlnet_aux](https://github.com/huggingface/controlnet_aux) | loads both of the above | Apache-2.0 | why no Drive checkpoint hunting is needed |
| [MobileCLIP2](https://github.com/apple/ml-mobileclip) | zero-shot labels + 512-d features | code MIT, **weights Apple ML Research Model License** | **research purposes only, no commercial use, no commercial products.** Features derived from it inherit that restriction — record it in your model card |
| [DINOv2](https://github.com/facebookresearch/dinov2) | 384-d shape features | Apache-2.0 (code and weights) | the specialised XRay/Cell variants are *not* Apache; this pipeline never loads them |
| [opennsfw2](https://github.com/bhaveshgohel/opennsfw2) | optional SFW screen | MIT | needs TensorFlow; Colab ships it |

Cite, if this dataset supports published work:

* Faghri *et al.*, **MobileCLIP2: Improving Multi-Modal Reinforced Training**, TMLR 2025
* Oquab *et al.*, **DINOv2: Learning Robust Visual Features without Supervision**, 2023
* Chan *et al.*, **Informative Drawings**, CVPR 2022
* Zhu *et al.*, **Anime2Sketch: A Sketch Extractor for Anime Arts with Deep Networks**, 2021

**Dataset licences are yours to verify.** Cell 4 prints what to check for each
preset and deliberately ships no `license_id`: a provenance manifest that guesses
at licences is worse than no manifest at all.